In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ==========================================================
# Configuration
# ==========================================================

BRONZE_TABLE = "finance_intelligence_hub.bronze.stock_prices_raw"
SILVER_TABLE = "finance_intelligence_hub.silver.stock_prices"

print("=" * 70)
print("Finance Intelligence Hub")
print("Silver Layer Pipeline - Stock Prices")
print("=" * 70)

# ==========================================================
# Create Silver Table if it does not exist
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {SILVER_TABLE}

(

ticker STRING,

trade_date DATE,

open_price double,

high_price double,

low_price double,

close_price double,

adjusted_close_price double,

volume BIGINT,

dwh_loaded_at TIMESTAMP,

silver_loaded_at TIMESTAMP

)

USING DELTA

""")

print("Silver table verified.")

# ==========================================================
# Incremental Load
# ==========================================================

last_loaded = spark.sql(

f"""SELECT COALESCE(MAX(dwh_loaded_at),TIMESTAMP('1900-01-01')) AS last_load FROM {SILVER_TABLE}"""

).first()["last_load"]

print(f"Last Silver Load : {last_loaded}")

bronze_df = (

spark.table(BRONZE_TABLE)

.filter(F.col("dwh_loaded_at") > F.lit(last_loaded))

)

rows = bronze_df.count()

print(f"New Bronze Records : {rows}")

if rows == 0:

    print("No new records found.")

    dbutils.notebook.exit("Silver layer already up to date.")

print("=" * 70)
# ==========================================================
# Data Cleaning & Transformation
# ==========================================================

clean_df = (

    bronze_df

    # Remove records with missing business keys
    .filter(F.col("ticker").isNotNull())
    .filter(F.col("date").isNotNull())

    # Standardize data types
    .withColumn("trade_date", F.to_date("date"))

    .withColumn("open_price", F.col("open").cast("double"))
    .withColumn("high_price", F.col("high").cast("double"))
    .withColumn("low_price", F.col("low").cast("double"))
    .withColumn("close_price", F.col("close").cast("double"))
    .withColumn("adjusted_close_price", F.col("adj_close").cast("double"))

    .withColumn("volume", F.col("volume").cast("double").cast("bigint"))

    .withColumn("dwh_loaded_at", F.col("dwh_loaded_at").cast("timestamp"))

    .withColumn(
        "silver_loaded_at",
        F.current_timestamp()
    )

)

print(f"Rows after type casting : {clean_df.count()}")

# ==========================================================
# Remove Invalid Market Data
# ==========================================================

clean_df = (

    clean_df

    .filter(F.col("open_price") > 0)
    .filter(F.col("high_price") > 0)
    .filter(F.col("low_price") > 0)
    .filter(F.col("close_price") > 0)
    .filter(F.col("adjusted_close_price") > 0)

    .filter(F.col("volume") >= 0)

)

print(f"Rows after removing invalid prices : {clean_df.count()}")

# ==========================================================
# OHLC Validation
# ==========================================================

clean_df = (

    clean_df

    .filter(F.col("high_price") >= F.col("open_price"))
    .filter(F.col("high_price") >= F.col("close_price"))
    .filter(F.col("high_price") >= F.col("low_price"))

    .filter(F.col("low_price") <= F.col("open_price"))
    .filter(F.col("low_price") <= F.col("close_price"))

)

print(f"Rows after OHLC validation : {clean_df.count()}")

# ==========================================================
# Remove Duplicate Records
# Keep only the latest version of each ticker/date
# ==========================================================

window_spec = (

    Window

    .partitionBy(
        "ticker",
        "trade_date"
    )

    .orderBy(
        F.col("dwh_loaded_at").desc()
    )

)

clean_df = (

    clean_df

    .withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )

    .filter(F.col("row_num") == 1)

    .drop("row_num")

)

print(f"Rows after deduplication : {clean_df.count()}")

# ==========================================================
# Select Final Silver Schema
# ==========================================================

silver_df = clean_df.select(

    "ticker",

    "trade_date",

    "open_price",

    "high_price",

    "low_price",

    "close_price",

    "adjusted_close_price",

    "volume",

    "dwh_loaded_at",

    "silver_loaded_at"

)

print("=" * 70)
print("Data Cleaning Completed Successfully")
print("=" * 70)

silver_df.show(10, truncate=False)
# ==========================================================
# MERGE INTO Silver Layer
# ==========================================================

print("=" * 70)
print("Starting MERGE into Silver Table...")
print("=" * 70)

delta_table = DeltaTable.forName(
    spark,
    SILVER_TABLE
)

(
    delta_table.alias("target")

    .merge(

        silver_df.alias("source"),

        """
        target.ticker = source.ticker
        AND
        target.trade_date = source.trade_date
        """

    )

    .whenMatchedUpdate(

        condition="""
        source.dwh_loaded_at > target.dwh_loaded_at
        """,

        set={

            "open_price": "source.open_price",
            "high_price": "source.high_price",
            "low_price": "source.low_price",
            "close_price": "source.close_price",
            "adjusted_close_price": "source.adjusted_close_price",
            "volume": "source.volume",
            "dwh_loaded_at": "source.dwh_loaded_at",
            "silver_loaded_at": "source.silver_loaded_at"

        }

    )

    .whenNotMatchedInsert(

        values={

            "ticker": "source.ticker",
            "trade_date": "source.trade_date",
            "open_price": "source.open_price",
            "high_price": "source.high_price",
            "low_price": "source.low_price",
            "close_price": "source.close_price",
            "adjusted_close_price": "source.adjusted_close_price",
            "volume": "source.volume",
            "dwh_loaded_at": "source.dwh_loaded_at",
            "silver_loaded_at": "source.silver_loaded_at"

        }

    )

    .execute()

)

print("MERGE completed successfully.")

# ==========================================================
# Delta Table Maintenance
# ==========================================================

print("Running OPTIMIZE...")

spark.sql(f"""

OPTIMIZE {SILVER_TABLE}

ZORDER BY (ticker, trade_date)

""")

print("OPTIMIZE completed.")

# ==========================================================
# Statistics
# ==========================================================

total_rows = spark.sql(

f"""

SELECT COUNT(*)

FROM {SILVER_TABLE}

"""

).first()[0]

total_tickers = spark.sql(

f"""SELECT COUNT(DISTINCT ticker) FROM {SILVER_TABLE}"""

).first()[0]

date_range = spark.sql(

f"""SELECT MIN(trade_date), MAX(trade_date) FROM {SILVER_TABLE}"""

).first()

print("\n" + "=" * 70)

print("Silver Pipeline Completed Successfully")

print("=" * 70)

print(f"Total Records  : {total_rows:,}")

print(f"Unique Tickers : {total_tickers:,}")

print(f"Date Range     : {date_range[0]} --> {date_range[1]}")

print("=" * 70)